# Weather Prediction using RNN and LSTM
This notebook demonstrates weather prediction for Cartagena using both RNN and LSTM models.

Basado en el Notebook de Farrukh Qureshi - Assistant Professor at Namal University Mianwali

In [1]:
!pip install tensorflow meteostat -qqq

In [6]:
# Import required libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import meteostat as ms

from datetime import date, datetime, timedelta
from sklearn.preprocessing import MinMaxScaler
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, LSTM, SimpleRNN
from tensorflow.keras.optimizers import Adam

# Set random seed for reproducibility
np.random.seed(42)

## Data Collection
Fetch weather data for Cartagena from Meteostat

In [9]:
# Cartagena coordinates
latitude = 10.3943121
longitude = -75.4789874

# Get data from Meteostat
start = date(2020, 1, 3)  # 5 years ago from current date
end = date(2025, 3, 11)    # Current date
POINT = ms.Point(latitude,longitude, 2)

stations = ms.stations.nearby(POINT, limit=4)

ts = ms.daily(stations, start, end)

data = ms.interpolate(ts, POINT).fetch()

# Display first few rows of the data
data.head()

,temp,tmin,tmax,rhum,prcp,wspd,pres,cldc
time,,,,,,,,
2020-01-03,27.7,24.2,32.2,81,<NA>,11.0,1011.7,<NA>
2020-01-04,27.3,23.4,31.0,84,0.0,11.1,1012.8,<NA>
2020-01-05,28.6,23.9,33.0,75,0.0,17.9,1013.6,<NA>
2020-01-06,28.2,26.0,33.5,76,0.0,18.3,1011.9,<NA>
2020-01-07,28.1,24.0,32.0,78,<NA>,14.5,1011.3,<NA>


## Data Preprocessing

In [10]:
# Prepare data (we'll use temperature as our target)
df = data[['temp']].copy()
df.ffill(inplace=True)  # Forward fill any missing values

# Scale the data
scaler = MinMaxScaler()
scaled_data = scaler.fit_transform(df)

# Create sequences for training
def create_sequences(data, seq_length):
    X, y = [], []
    for i in range(len(data) - seq_length):
        X.append(data[i:(i + seq_length), 0])
        y.append(data[i + seq_length, 0])
    return np.array(X), np.array(y)

# Parameters
sequence_length = 30
X, y = create_sequences(scaled_data, sequence_length)

# Split data into train and test
train_size = len(X) - 30  # Last 30 days for comparison
X_train, X_test = X[:train_size], X[train_size:]
y_train, y_test = y[:train_size], y[train_size:]

# Reshape input for LSTM/RNN [samples, time steps, features]
X_train = X_train.reshape((X_train.shape[0], X_train.shape[1], 1))
X_test = X_test.reshape((X_test.shape[0], X_test.shape[1], 1))

## Model Creation and Training

In [11]:
# Create RNN model
def create_rnn_model():
    model = Sequential([
        SimpleRNN(50, input_shape=(sequence_length, 1)),
        Dense(1)
    ])
    model.compile(optimizer=Adam(learning_rate=0.001), loss='mse')
    return model

# Create LSTM model
def create_lstm_model():
    model = Sequential([
        LSTM(50, input_shape=(sequence_length, 1)),
        Dense(1)
    ])
    model.compile(optimizer=Adam(learning_rate=0.001), loss='mse')
    return model

# Train models
rnn_model = create_rnn_model()
lstm_model = create_lstm_model()

print("Training RNN model...")
rnn_history = rnn_model.fit(X_train, y_train, epochs=50, batch_size=32, verbose=1)

print("\nTraining LSTM model...")
lstm_history = lstm_model.fit(X_train, y_train, epochs=50, batch_size=32, verbose=1)

I0000 00:00:1773418209.026450     172 gpu_device.cc:2043] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 3540 MB memory:  -> device: 0, name: NVIDIA RTX 1000 Ada Generation Laptop GPU, pci bus id: 0000:01:00.0, compute capability: 8.9
/usr/local/lib/python3.11/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Training RNN model...
Epoch 1/50


I0000 00:00:1773418211.274911     350 service.cc:153] XLA service 0x7a2bf40fd4f0 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1773418211.275000     350 service.cc:161]   StreamExecutor [0]: NVIDIA RTX 1000 Ada Generation Laptop GPU, Compute Capability 8.9 (Driver: 13.0.0; Runtime: 12.8.0; Toolkit: 12.5.0; DNN: 9.10.2)
I0000 00:00:1773418211.338433     350 dump_mlir_util.cc:269] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
I0000 00:00:1773418211.514110     350 cuda_dnn.cc:461] Loaded cuDNN version 91002
I0000 00:00:1773418211.542948     350 dot_merger.cc:481] Merging Dots in computation: sequential_1_simple_rnn_1_while_body_1160_grad_1352_const_0__.15.clone.clone.clone.clone.clone


14/58 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - loss: 0.3118

I0000 00:00:1773418212.988324     350 device_compiler.h:208] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


56/58 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - loss: 0.1333

I0000 00:00:1773418213.813727     352 dot_merger.cc:481] Merging Dots in computation: sequential_1_simple_rnn_1_while_body_1160_grad_1352_const_0__.15.clone.clone.clone.clone.clone


58/58 ━━━━━━━━━━━━━━━━━━━━ 5s 37ms/step - loss: 0.0478
Epoch 2/50
58/58 ━━━━━━━━━━━━━━━━━━━━ 1s 12ms/step - loss: 0.0129
Epoch 3/50
58/58 ━━━━━━━━━━━━━━━━━━━━ 1s 14ms/step - loss: 0.0111
Epoch 4/50
58/58 ━━━━━━━━━━━━━━━━━━━━ 1s 13ms/step - loss: 0.0107
Epoch 5/50
58/58 ━━━━━━━━━━━━━━━━━━━━ 1s 14ms/step - loss: 0.0094
Epoch 6/50
58/58 ━━━━━━━━━━━━━━━━━━━━ 1s 15ms/step - loss: 0.0088
Epoch 7/50
58/58 ━━━━━━━━━━━━━━━━━━━━ 1s 15ms/step - loss: 0.0102
Epoch 8/50
58/58 ━━━━━━━━━━━━━━━━━━━━ 1s 15ms/step - loss: 0.0086
Epoch 9/50
58/58 ━━━━━━━━━━━━━━━━━━━━ 1s 15ms/step - loss: 0.0086
Epoch 10/50
58/58 ━━━━━━━━━━━━━━━━━━━━ 1s 16ms/step - loss: 0.0091
Epoch 11/50
58/58 ━━━━━━━━━━━━━━━━━━━━ 1s 19ms/step - loss: 0.0080
Epoch 12/50
58/58 ━━━━━━━━━━━━━━━━━━━━ 1s 16ms/step - loss: 0.0079
Epoch 13/50
58/58 ━━━━━━━━━━━━━━━━━━━━ 1s 17ms/step - loss: 0.0080
Epoch 14/50
58/58 ━━━━━━━━━━━━━━━━━━━━ 1s 23ms/step - loss: 0.0081
Epoch 15/50
58/58 ━━━━━━━━━━━━━━━━━━━━ 1s 18ms/step - loss: 0.0080
Epoch 16/50
58/

## Model Evaluation

In [ ]:
# Make predictions
rnn_pred = rnn_model.predict(X_test)
lstm_pred = lstm_model.predict(X_test)

# Inverse transform predictions
rnn_pred = scaler.inverse_transform(rnn_pred)
lstm_pred = scaler.inverse_transform(lstm_pred)
y_test_actual = scaler.inverse_transform([y_test]).T

# Calculate error metrics
def calculate_metrics(actual, predicted):
    mse = np.mean((actual - predicted) ** 2)
    rmse = np.sqrt(mse)
    mae = np.mean(np.abs(actual - predicted))
    return mse, rmse, mae

rnn_metrics = calculate_metrics(y_test_actual, rnn_pred)
lstm_metrics = calculate_metrics(y_test_actual, lstm_pred)

# Print metrics
print("\nRNN Metrics:")
print(f"MSE: {rnn_metrics[0]:.2f}")
print(f"RMSE: {rnn_metrics[1]:.2f}")
print(f"MAE: {rnn_metrics[2]:.2f}")

print("\nLSTM Metrics:")
print(f"MSE: {lstm_metrics[0]:.2f}")
print(f"RMSE: {lstm_metrics[1]:.2f}")
print(f"MAE: {lstm_metrics[2]:.2f}")

## Visualization

In [ ]:
# Plot results
plt.figure(figsize=(12, 6))
plt.plot(y_test_actual, label='Actual')
plt.plot(rnn_pred, label='RNN Predictions')
plt.plot(lstm_pred, label='LSTM Predictions')
plt.title('Last 30 Days: Actual vs Predicted Temperature')
plt.xlabel('Days')
plt.ylabel('Temperature (°C)')
plt.legend()
plt.grid(True)
plt.show()

## Future Predictions

In [ ]:
# Predict next 5 days
last_sequence = scaled_data[-sequence_length:]
last_sequence = last_sequence.reshape(1, sequence_length, 1)

future_rnn = []
future_lstm = []

for _ in range(5):
    # RNN prediction
    rnn_next = rnn_model.predict(last_sequence)
    future_rnn.append(rnn_next[0, 0])

    # LSTM prediction
    lstm_next = lstm_model.predict(last_sequence)
    future_lstm.append(lstm_next[0, 0])

    # Update sequence
    last_sequence = np.roll(last_sequence, -1)
    last_sequence[0, -1, 0] = lstm_next[0, 0]

# Convert predictions to temperature values
future_rnn = scaler.inverse_transform(np.array(future_rnn).reshape(-1, 1))
future_lstm = scaler.inverse_transform(np.array(future_lstm).reshape(-1, 1))

print("\nNext 5 days temperature predictions:")
for i in range(5):
    print(f"Day {i+1}:")
    print(f"RNN: {future_rnn[i][0]:.2f}°C")
    print(f"LSTM: {future_lstm[i][0]:.2f}°C")